# 02 — Preprocessing: Clean, Dedupe, Split, Weighted Sampling

From `01_eda.ipynb` we know:
- `lang` needs normalizing (`CPP`/`C++` → `C++`)
- The label is heavily imbalanced (`vul=1` is the minority, roughly 1:17)
- There are exact duplicate `func_before` rows

This notebook turns the raw HuggingFace dataset into three clean, deduplicated,
leakage-free files: `data/processed/train.parquet`, `val.parquet`, `test.parquet`,
plus `class_weights.json` for weighted sampling in Phase 4.

In [1]:
import json
from pathlib import Path

import pandas as pd
from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

## 1. Load and combine all splits

The dataset ships with pre-made `train`/`validation`/`test` splits, but we
**don't use them as-is**: they were created before deduplication, so the same
function can appear in more than one of them. If we kept their split
boundaries, a duplicate could end up in both train and test — the model
would then be evaluated on code it already memorized (data leakage), making
test accuracy look better than it really is.

Instead we combine everything, deduplicate globally, and create our own
stratified split.

In [2]:
dsd = load_dataset("benjis/bigvul")
full = concatenate_datasets([dsd["train"], dsd["validation"], dsd["test"]])
df = full.to_pandas()
print("Combined raw rows:", len(df))

Combined raw rows: 217007


## 2. Normalize language and filter to C/C++

In [3]:
LANG_MAP = {"C": "C", "CPP": "C++", "C++": "C++"}
df["lang"] = df["lang"].map(LANG_MAP)
df = df[df["lang"].isin(["C", "C++"])].copy()
print("After lang filter:", len(df))
print(df["lang"].value_counts())

After lang filter: 217007
lang
C      213919
C++      3088
Name: count, dtype: int64


## 3. Clean: strip whitespace, drop empty functions

In [4]:
df["func_before"] = df["func_before"].astype(str).str.strip()
before = len(df)
df = df[df["func_before"].str.len() > 0]
print(f"Dropped {before - len(df)} rows with empty func_before")

Dropped 0 rows with empty func_before


## 4. Deduplicate

We dedupe on the exact text of `func_before` — this is the field we'll train
the classifier on, so two rows with identical `func_before` are effectively
the same training example even if their CVE/commit metadata differs.

In [5]:
before = len(df)
df = df.drop_duplicates(subset=["func_before"], keep="first").reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows ({(before - len(df)) / before:.1%})")
print(f"Remaining: {len(df)} rows")

vc = df["vul"].value_counts()
print(f"\nClass balance after dedup: not-vul={vc[0]}, vul={vc[1]}, ratio={vc[0]/vc[1]:.1f}:1")

Dropped 51572 duplicate rows (23.8%)
Remaining: 165435 rows

Class balance after dedup: not-vul=156271, vul=9164, ratio=17.1:1


## 5. Stratified train/val/test split (80/10/10)

We stratify on `vul` **and** `lang` combined, so both the label balance and
the C/C++ ratio stay consistent across train/val/test — otherwise, since
C++ is a small minority of the data, random chance could easily skew it
into one split more than another.

In [6]:
df["strata"] = df["vul"].astype(str) + "_" + df["lang"]
print(df["strata"].value_counts())

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["strata"], random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["strata"], random_state=RANDOM_STATE
)

print(f"\ntrain: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

strata
0_C      154011
1_C        9067
0_C++      2260
1_C++        97
Name: count, dtype: int64



train: 132348  val: 16543  test: 16544


In [7]:
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = split["vul"].value_counts()
    print(f"{name:5s} -> not-vul={vc[0]:6d}  vul={vc[1]:5d}  ratio={vc[0]/vc[1]:.2f}:1")

train -> not-vul=125017  vul= 7331  ratio=17.05:1
val   -> not-vul= 15627  vul=  916  ratio=17.06:1
test  -> not-vul= 15627  vul=  917  ratio=17.04:1


## 6. Class weights for weighted sampling

**Why weighted sampling?** With ~17 "not vulnerable" examples for every
"vulnerable" one, a model can reach >94% accuracy by always predicting
"not vulnerable" — technically high accuracy, but useless in practice. A
`WeightedRandomSampler` fixes this at training time: instead of drawing
training examples uniformly at random, it draws minority-class (vulnerable)
examples much more often, so each training batch sees roughly balanced
classes.

We compute the weight here (Phase 3, from the train split) and consume it
in Phase 4's `DataLoader`.

In [8]:
class_counts = train_df["vul"].value_counts().to_dict()
# Inverse-frequency weighting: rarer class gets a proportionally larger weight.
class_weights = {str(c): len(train_df) / (2 * n) for c, n in class_counts.items()}
print("Class weights:", class_weights)

with open(PROCESSED_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=2)
print(f"Saved to {PROCESSED_DIR / 'class_weights.json'}")

Class weights: {'0': 0.529320012478303, '1': 9.026599372527622}
Saved to ..\data\processed\class_weights.json


### Quick demo — how this plugs into `WeightedRandomSampler` (used for real in Phase 4)

In [9]:
import torch
from torch.utils.data import WeightedRandomSampler

sample_weights = train_df["vul"].map(lambda v: class_weights[str(v)]).to_numpy()
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

# Simulate one epoch of sampling and check the effective class balance.
sampled_labels = train_df["vul"].to_numpy()[list(sampler)]
print("Effective class balance when sampled with weights:")
print(pd.Series(sampled_labels).value_counts())

Effective class balance when sampled with weights:
1    66217
0    66131
Name: count, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_19940\3310490215.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights=torch.as_tensor(sample_weights, dtype=torch.double),


## 7. Save processed splits

In [10]:
KEEP_COLS = ["project", "commit_id", "CWE ID", "lang", "func_before", "func_after", "vul"]

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    out = split[KEEP_COLS].reset_index(drop=True)
    out.to_parquet(PROCESSED_DIR / f"{name}.parquet", index=False)
    print(f"Saved {name}.parquet: {out.shape}")

Saved train.parquet: (132348, 7)
Saved val.parquet: (16543, 7)
Saved test.parquet: (16544, 7)


## 8. Sanity check — reload from disk

In [11]:
for name in ["train", "val", "test"]:
    reloaded = pd.read_parquet(PROCESSED_DIR / f"{name}.parquet")
    print(name, reloaded.shape, dict(reloaded["vul"].value_counts()))

with open(PROCESSED_DIR / "class_weights.json") as f:
    print("\nclass_weights.json:", json.load(f))

train (132348, 7) {0: np.int64(125017), 1: np.int64(7331)}
val (16543, 7) {0: np.int64(15627), 1: np.int64(916)}
test (16544, 7) {0: np.int64(15627), 1: np.int64(917)}

class_weights.json: {'0': 0.529320012478303, '1': 9.026599372527622}


## Summary — what's in `data/processed/` now

- `train.parquet`, `val.parquet`, `test.parquet` — cleaned, deduplicated, stratified 80/10/10 split, no leakage between them
- `class_weights.json` — inverse-frequency class weights, ready for `WeightedRandomSampler` in Phase 4

Next: **Phase 4** loads `train.parquet` / `val.parquet`, tokenizes `func_before` with the GraphCodeBERT tokenizer, and fine-tunes a binary classifier using the weighted sampler set up here.